# Lab 3: Functions II
**Difficulty: Intermediate | ~25 min | Requires Lab 1 and Lab 2**

## Setup

In [1]:
# This lab uses only Python's built-in features, but this cell keeps the
# notebook runnable from a fresh kernel. It installs nothing extra.
!python -m pip install --upgrade pip

## Accept Any Number of Positional Arguments with `*args`

In [2]:
# *amounts collects any number of positional float arguments into one tuple.
# The signature stays the same whether we pass 3 values, 5 values, or none.

def total_sales(*amounts: float) -> float:
    # sum(...) works directly on the collected tuple of amounts.
    return sum(amounts)


# A single function call handles three values, five values, or none at all.
print(f"Total of 3 sales: {total_sales(25.0, 40.0, 15.5)}")
print(f"Total of 5 sales: {total_sales(10, 20, 30, 40, 50)}")

# No arguments -> amounts is (), and sum(()) returns 0.
print(f"Total of 0 sales: {total_sales()}")  # empty tuple sums to 0


Total of 3 sales: 80.5
Total of 5 sales: 150
Total of 0 sales: 0


## Accept Any Number of Named Arguments with `**kwargs`

In [3]:
# **details collects any number of keyword=value pairs into a dictionary.
# We never declare the field names in advance -- callers decide them.

def build_product_label(**details: str) -> str:
    # Build one "key: value" fragment per keyword argument.
    parts = [f"{key}: {value}" for key, value in details.items()]

    # Join every fragment with a comma + space into one label string.
    return ", ".join(parts)


# Each caller passes a completely different set of field names.
label_a = build_product_label(name="Mug", color="Blue", price="$8")
label_b = build_product_label(name="Notebook", pages="200")


print(label_a)
print(label_b)


name: Mug, color: Blue, price: $8
name: Notebook, pages: 200


## Write Lambda Expressions, and Sort with One

In [4]:
# lambda: a tiny anonymous function, useful for short one-off expressions.
# "lambda x: x ** 2" is shorthand for "def square(x): return x ** 2" --
# except the value of the single expression is returned automatically.

square = lambda x: x ** 2   # one parameter
add = lambda a, b: a + b    # two parameters


print(f"square(5) = {square(5)}")
print(f"add(3, 4) = {add(3, 4)}")


# Our unsorted list of product records.
products = [
    {"name": "Mug", "price": 8.0},
    {"name": "Notebook", "price": 3.5},
    {"name": "Pen", "price": 1.25},
]

# sorted(key=...) accepts a function that decides what to compare on.
# The lambda returns each item's "price", so we sort by price value.
sorted_by_price = sorted(products, key=lambda item: item["price"])

# Print each product in sorted order, cheapest first.
for product in sorted_by_price:
    print(f"{product['name']}: ${product['price']}")


square(5) = 25
add(3, 4) = 7
Pen: $1.25
Notebook: $3.5
Mug: $8.0


## Transform with `map()`, Keep Matches with `filter()`

In [5]:
# Our raw list of prices to process.
prices = [8.0, 3.5, 1.25, 12.0]

# map() applies a function to EVERY element and returns an iterator of results.
# Here the lambda discounts each price by 10% and rounds to 2 decimals.
discounted_prices = list(map(lambda p: round(p * 0.9, 2), prices))

# map/filter return a lazy iterator, so we wrap each in list(...) to print it.
print(f"Original prices: {prices}")
print(f"Discounted prices (10% off): {discounted_prices}")


# filter() keeps only the elements for which the predicate returns True.
# Here the lambda keeps only prices strictly under $5.
affordable_prices = list(filter(lambda p: p < 5.0, prices))
print(f"Prices under $5: {affordable_prices}")


Original prices: [8.0, 3.5, 1.25, 12.0]
Discounted prices (10% off): [7.2, 3.15, 1.12, 10.8]
Prices under $5: [3.5, 1.25]


## Combine Everything into an Order-Summary Function

In [6]:
# *args and **kwargs can be combined in a single signature.
# Positional values land in item_prices; named values land in order_info.

def summarize_order(*item_prices: float, **order_info: str) -> str:
    # Sum all the positional prices to get the subtotal.
    subtotal = sum(item_prices)

    # .get(key, default) returns the value, or the default if the key is
    # missing -- so calls that omit a detail still work safely.
    customer = order_info.get("customer", "Guest")
    priority = order_info.get("priority", "standard")
    return f"Order for {customer} ({priority}): {len(item_prices)} item(s), subtotal ${subtotal:.2f}"


# First call passes both named details; second call relies on the defaults.
print(summarize_order(8.0, 3.5, 1.25, customer="Ana", priority="rush"))
print(summarize_order(12.0, customer="Ben"))


# A fuller product catalogue, including whether each item is in stock.
products_full = [
    {"name": "Mug", "price": 8.0, "in_stock": True},
    {"name": "Notebook", "price": 3.5, "in_stock": False},
    {"name": "Pen", "price": 1.25, "in_stock": True},
    {"name": "Backpack", "price": 45.0, "in_stock": True},
]

# Chain filter -> sort -> map to build a clean one-way pipeline.

# Step 1: filter() keeps only the products that are in stock.
in_stock_products = list(filter(lambda p: p["in_stock"], products_full))

# Step 2: sorted(key=...) reorders that smaller list by price, cheapest first.
cheapest_first = sorted(in_stock_products, key=lambda p: p["price"])

# Step 3: map() extracts just each product's name into a new list.
product_names = list(map(lambda p: p["name"], cheapest_first))

print(f"In-stock products, cheapest first: {product_names}")


Order for Ana (rush): 3 item(s), subtotal $12.75
Order for Ben (standard): 1 item(s), subtotal $12.00
In-stock products, cheapest first: ['Pen', 'Mug', 'Backpack']
